In [2]:
import sys
import setuptools._distutils.version
sys.modules['distutils.version'] = setuptools._distutils.version

import re
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── All category URLs ──────────────────────────────────────────────────────────
URLS = [
    "https://www.carousell.com.my/categories/following-1/",
    "https://www.carousell.com.my/categories/free-items-2156/",
    "https://www.carousell.com.my/categories/computers-tech-602/",
    "https://www.carousell.com.my/categories/computers-tech-602/desktops-1794/",
    "https://www.carousell.com.my/categories/computers-tech-602/laptops-notebooks-1793/",
    "https://www.carousell.com.my/categories/computers-tech-602/parts-accessories-604/",
    "https://www.carousell.com.my/categories/computers-tech-602/parts-accessories-604/cables-adaptors-5020/",
    "https://www.carousell.com.my/categories/computers-tech-602/parts-accessories-604/chargers-5072/",
    "https://www.carousell.com.my/categories/computers-tech-602/parts-accessories-604/mouse-mousepads-5074/",
    "https://www.carousell.com.my/categories/computers-tech-602/parts-accessories-604/monitor-screens-5075/",
    "https://www.carousell.com.my/categories/computers-tech-602/parts-accessories-604/computer-keyboard-5076/",
    "https://www.carousell.com.my/categories/computers-tech-602/parts-accessories-604/hard-disks-thumbdrives-5077/",
    "https://www.carousell.com.my/categories/computers-tech-602/parts-accessories-604/networking-5091/",
    "https://www.carousell.com.my/categories/computers-tech-602/parts-accessories-604/computer-parts-5113/",
    "https://www.carousell.com.my/categories/computers-tech-602/parts-accessories-604/software-5188/",
    "https://www.carousell.com.my/categories/computers-tech-602/parts-accessories-604/webcams-5196/",
    "https://www.carousell.com.my/categories/computers-tech-602/parts-accessories-604/laptop-bags-sleeves-5207/",
    "https://www.carousell.com.my/categories/computers-tech-602/parts-accessories-604/other-accessories-609/",
    "https://www.carousell.com.my/categories/computers-tech-602/printers-scanners-copiers-5226/",
    "https://www.carousell.com.my/categories/computers-tech-602/office-business-technology-5247/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/iphones-1248/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/iphones-1248/iphone-air-series-6754/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/iphones-1248/iphone-17-series-6753/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/iphones-1248/iphone-16-series-6383/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/iphones-1248/iphone-15-series-6366/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/iphones-1248/iphone-14-series-6330/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/iphones-1248/iphone-13-series-2243/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/iphones-1248/iphone-se-series-5253/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/iphones-1248/iphone-12-series-5408/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/iphones-1248/iphone-11-series-2148/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/iphones-1248/iphone-x-series-1632/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/iphones-1248/iphone-8-series-1617/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/iphones-1248/iphone-7-series-1275/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/iphones-1248/iphone-6-series-1274/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/iphones-1248/iphone-others-1276/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/samsung-1267/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/sony-1268/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/xiaomi-1269/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/lg-1270/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/oppo-1271/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/google-pixel-5254/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/vivo-5264/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/huawei-5283/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/oneplus-5292/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/motorola-5690/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/realme-5966/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/asus-5967/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/nothing-6378/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/honor-6379/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/infinix-6380/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/android-phones-1249/android-others-1272/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-phones-5251/early-generation-mobile-phones-5989/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/tablets-1250/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/tablets-1250/ipad-5990/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/tablets-1250/android-5993/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/tablets-1250/windows-5994/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/tablets-1250/others-5995/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/e-readers-5996/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/wearables-smart-watches-5997/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-gadget-accessories-606/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-gadget-accessories-606/batteries-power-banks-1853/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-gadget-accessories-606/cases-covers-1852/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-gadget-accessories-606/chargers-cables-5998/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-gadget-accessories-606/mounts-holders-5999/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-gadget-accessories-606/sim-cards-6000/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-gadget-accessories-606/memory-sd-cards-6001/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/mobile-gadget-accessories-606/other-mobile-gadget-accessories-1854/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/walkie-talkie-6002/",
    "https://www.carousell.com.my/categories/mobile-phones-gadgets-605/other-gadgets-1251/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/activewear-6129/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/maternity-wear-6130/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/tops-1261/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/tops-1261/blouses-6131/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/tops-1261/shirts-6132/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/tops-1261/sleeveless-6133/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/tops-1261/longsleeves-6134/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/tops-1261/other-tops-6135/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/bottoms-1262/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/bottoms-1262/jeans-leggings-6136/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/bottoms-1262/shorts-6137/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/bottoms-1262/skirts-6138/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/bottoms-1262/other-bottoms-6139/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/dresses-sets-1259/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/dresses-sets-1259/dresses-6140/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/dresses-sets-1259/evening-dresses-gowns-617/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/dresses-sets-1259/jumpsuits-6141/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/dresses-sets-1259/rompers-1264/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/dresses-sets-1259/sets-or-coordinates-6142/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/dresses-sets-1259/traditional-ethnic-wear-6143/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/footwear-612/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/footwear-612/boots-6144/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/footwear-612/flats-6145/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/footwear-612/shoe-inserts-6146/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/footwear-612/heels-6147/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/footwear-612/loafers-6148/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/footwear-612/sandals-6149/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/footwear-612/sneakers-6150/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/footwear-612/wedges-6151/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/footwear-612/flipflops-and-slides-6152/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/swimwear-6153/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/swimwear-6153/bikinis-swimsuits-6154/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/swimwear-6153/rash-guard-6155/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/swimwear-6153/muslimah-swimwear-6307/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/muslimah-fashion-66/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/muslimah-fashion-66/baju-kurung-sets-6156/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/muslimah-fashion-66/kaftans-jubahs-6157/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/muslimah-fashion-66/hijabs-1623/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/muslimah-fashion-66/tops-1620/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/muslimah-fashion-66/dresses-1622/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/muslimah-fashion-66/bottoms-1621/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/muslimah-fashion-66/prayer-sets-6158/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/muslimah-fashion-66/accessories-6159/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/coats-jackets-and-outerwear-1263/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/bags-wallets-610/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/bags-wallets-610/purses-pouches-6160/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/bags-wallets-610/clutches-6161/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/bags-wallets-610/shoulder-bags-6162/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/bags-wallets-610/backpacks-6163/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/bags-wallets-610/cross-body-bags-6164/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/bags-wallets-610/beach-bags-6165/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/bags-wallets-610/tote-bags-6166/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/bags-wallets-610/wallets-card-holders-6167/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/jewelry-organisers-613/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/jewelry-organisers-613/rings-6168/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/jewelry-organisers-613/earrings-6169/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/jewelry-organisers-613/charms-6170/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/jewelry-organisers-613/bracelets-6171/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/jewelry-organisers-613/necklaces-6172/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/jewelry-organisers-613/anklets-6173/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/jewelry-organisers-613/body-jewelry-6174/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/jewelry-organisers-613/precious-stones-6175/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/jewelry-organisers-613/brooches-6176/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/jewelry-organisers-613/accessory-holder-box-organisers-6177/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/watches-accessories-615/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/watches-accessories-615/belts-6178/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/watches-accessories-615/watches-614/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/watches-accessories-615/hair-accessories-6179/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/watches-accessories-615/hats-beanies-6180/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/watches-accessories-615/gloves-6181/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/watches-accessories-615/scarves-6182/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/watches-accessories-615/socks-tights-6183/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/watches-accessories-615/sunglasses-eyewear-6184/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/watches-accessories-615/other-accessories-6185/",
    "https://www.carousell.com.my/categories/women-s-fashion-4/new-undergarments-loungewear-6186/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/activewear-6187/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/tops-sets-1848/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/tops-sets-1848/tshirts-polo-shirts-6188/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/tops-sets-1848/hoodies-6189/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/tops-sets-1848/formal-shirts-6190/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/tops-sets-1848/swim-top-rash-guards-6191/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/tops-sets-1848/vests-6192/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/tops-sets-1848/sets-coordinates-6193/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/tops-sets-1848/sleep-and-loungewear-6194/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/bottoms-1849/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/bottoms-1849/jeans-6195/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/bottoms-1849/joggers-6196/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/bottoms-1849/shorts-6197/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/bottoms-1849/trousers-6198/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/bottoms-1849/chinos-6199/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/bottoms-1849/swim-trunks-board-shorts-6200/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/bottoms-1849/new-underwear-6201/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/bottoms-1849/sleep-and-loungewear-6202/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/footwear-620/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/footwear-620/sneakers-1832/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/footwear-620/casual-shoes-6203/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/footwear-620/dress-shoes-1831/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/footwear-620/flipflops-and-slides-1833/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/footwear-620/boots-1834/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/footwear-620/shoe-inserts-accessories-6204/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/muslim-wear-6205/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/muslim-wear-6205/tops-6206/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/muslim-wear-6205/baju-melayu-6207/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/muslim-wear-6205/sarong-6208/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/muslim-wear-6205/jubahs-6209/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/muslim-wear-6205/accessories-6210/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/coats-jackets-and-outerwear-1850/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/bags-618/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/bags-618/backpacks-1837/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/bags-618/briefcases-1838/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/bags-618/sling-bags-1839/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/bags-618/belt-bags-clutches-and-pouches-1840/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/watches-accessories-622/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/watches-accessories-622/watches-621/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/watches-accessories-622/cap-hats-1845/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/watches-accessories-622/jewelry-6211/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/watches-accessories-622/beanie-6212/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/watches-accessories-622/sunglasses-eyewear-1844/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/watches-accessories-622/belts-1842/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/watches-accessories-622/wallets-card-holders-6213/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/watches-accessories-622/ties-1843/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/watches-accessories-622/scarves-6214/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/watches-accessories-622/gloves-6215/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/watches-accessories-622/handkerchief-pocket-squares-6216/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/watches-accessories-622/cuff-links-6217/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/watches-accessories-622/socks-1846/",
    "https://www.carousell.com.my/categories/men-s-fashion-3/watches-accessories-622/accessory-holder-box-organisers-6218/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/sanitisers-disinfectants-6255/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/hands-nails-586/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/ear-care-6256/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/vision-care-6257/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/foot-care-6258/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/oral-care-6259/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/sanitary-hygiene-6260/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/fragrance-deodorants-6261/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/bath-body-583/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/bath-body-583/bath-6262/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/bath-body-583/body-care-6263/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/bath-body-583/hair-removal-6264/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/face-6265/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/face-6265/face-care-6266/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/face-6265/makeup-584/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/hair-582/",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/mens-grooming-585/",
    "https://www.carousell.com.my/categories/luxury-20/",
    "https://www.carousell.com.my/categories/luxury-20/bags-and-wallets-761/",
    "https://www.carousell.com.my/categories/luxury-20/luxury-apparel-760/",
    "https://www.carousell.com.my/categories/luxury-20/luxury-accessories-759/",
    "https://www.carousell.com.my/categories/luxury-20/luxury-watches-762/",
    "https://www.carousell.com.my/categories/luxury-20/sneakers-footwear-6305/",
    "https://www.carousell.com.my/categories/video-gaming-697/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-consoles-699/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-consoles-699/playstation-6003/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-consoles-699/nintendo-6004/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-consoles-699/xbox-6005/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-consoles-699/others-6006/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-700/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-700/playstation-6007/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-700/nintendo-6008/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-700/xbox-6009/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-700/others-6010/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/controllers-6011/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/cases-covers-6012/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/cables-chargers-6013/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/virtual-reality-6014/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/interactive-gaming-figures-6015/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/in-game-products-6016/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/game-gift-cards-accounts-6017/",
    "https://www.carousell.com.my/categories/audio-595/",
    "https://www.carousell.com.my/categories/audio-595/earphones-6018/",
    "https://www.carousell.com.my/categories/audio-595/headphones-headsets-6019/",
    "https://www.carousell.com.my/categories/audio-595/microphones-6020/",
    "https://www.carousell.com.my/categories/audio-595/voice-recorders-6021/",
    "https://www.carousell.com.my/categories/audio-595/portable-music-players-6022/",
    "https://www.carousell.com.my/categories/audio-595/portable-audio-accessories-6023/",
    "https://www.carousell.com.my/categories/audio-595/soundbars-speakers-amplifiers-6024/",
    "https://www.carousell.com.my/categories/audio-595/other-audio-equipment-6025/",
    "https://www.carousell.com.my/categories/photography-6/",
    "https://www.carousell.com.my/categories/photography-6/cameras-6026/",
    "https://www.carousell.com.my/categories/photography-6/lens-kits-6027/",
    "https://www.carousell.com.my/categories/photography-6/drones-6028/",
    "https://www.carousell.com.my/categories/photography-6/video-cameras-6029/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/camera-bags-carriers-6031/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/gimbals-stabilisers-6032/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/lighting-studio-equipment-6033/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/flashes-6034/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/tripods-monopods-6035/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/dry-boxes-cabinets-6036/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/batteries-chargers-6037/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/other-photography-accessories-6038/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/furniture-745/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/furniture-745/bed-frames-mattresses-6039/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/furniture-745/sofas-6040/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/furniture-745/tv-consoles-6041/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/furniture-745/tables-sets-6042/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/furniture-745/chairs-6043/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/furniture-745/shelves-cabinets-racks-6044/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/furniture-745/other-home-furniture-1258/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/outdoor-furniture-6045/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/lighting-fans-6046/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/lighting-fans-6046/lighting-6047/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/lighting-fans-6046/fans-6048/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-decor-1266/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-decor-1266/mirrors-6049/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-decor-1266/clocks-6050/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-decor-1266/wall-decor-6051/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-decor-1266/frames-pictures-6052/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-decor-1266/vases-decorative-bowls-6053/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-decor-1266/carpets-mats-flooring-6054/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-decor-1266/curtains-blinds-6055/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-decor-1266/cushions-throws-6056/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-decor-1266/artificial-plants-flowers-6057/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-decor-1266/other-home-decor-6058/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-fragrance-6059/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/bedding-towels-6060/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/bathroom-kitchen-fixtures-6061/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-improvement-organisation-6062/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-improvement-organisation-6062/ladders-steps-6063/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-improvement-organisation-6062/hooks-hangers-6064/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-improvement-organisation-6062/storage-boxes-baskets-6065/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-improvement-organisation-6062/clothes-drying-rack-6066/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-improvement-organisation-6062/laundry-baskets-6067/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/home-improvement-organisation-6062/home-improvement-tools-accessories-6068/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/cleaning-homecare-supplies-6069/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/cleaning-homecare-supplies-6069/detergents-6070/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/cleaning-homecare-supplies-6069/waste-bins-bags-6071/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/cleaning-homecare-supplies-6069/pest-control-6072/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/cleaning-homecare-supplies-6069/ironing-boards-6073/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/cleaning-homecare-supplies-6069/cleaning-tools-supplies-6074/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/kitchenware-tableware-741/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/kitchenware-tableware-741/cookware-accessories-6075/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/kitchenware-tableware-741/bakeware-6076/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/kitchenware-tableware-741/knives-chopping-boards-6077/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/kitchenware-tableware-741/food-organisation-storage-6078/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/kitchenware-tableware-741/dinnerware-cutlery-6079/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/kitchenware-tableware-741/pitchers-dispensers-6080/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/kitchenware-tableware-741/coffee-tea-tableware-6081/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/kitchenware-tableware-741/towels-napkins-holders-6082/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/kitchenware-tableware-741/table-linen-textiles-6083/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/kitchenware-tableware-741/water-bottles-tumblers-6084/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/kitchenware-tableware-741/other-kitchenware-tableware-6085/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/security-locks-6086/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/security-locks-6086/locks-doors-gates-6088/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/security-locks-6086/security-systems-cctv-cameras-6087/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/security-locks-6086/safe-6089/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/security-locks-6086/peephole-viewers-doorbells-6090/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/security-locks-6086/other-security-devices-6091/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/gardening-and-plants-746/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/gardening-and-plants-746/plants-seeds-6092/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/gardening-and-plants-746/pots-planters-6093/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/gardening-and-plants-746/gardening-tools-ornaments-6094/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/gardening-and-plants-746/garden-soil-fertilisers-6095/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/gardening-and-plants-746/grass-mowers-trimmers-6096/",
    "https://www.carousell.com.my/categories/furniture-home-living-13/gardening-and-plants-746/hose-and-watering-devices-6097/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/tv-entertainment-608/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/tv-entertainment-608/tv-6099/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/tv-entertainment-608/projectors-6100/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/tv-entertainment-608/entertainment-systems-smart-home-devices-6101/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/tv-entertainment-608/blu-ray-media-players-6102/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/tv-entertainment-608/media-streamers-hubs-6103/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/tv-entertainment-608/tv-parts-accessories-6104/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/refrigerators-freezers-6106/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/wine-cellar-storage-6107/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/hobs-hoods-6108/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/dishwasher-6109/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/ovens-toasters-6110/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/cookers-6111/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/coffee-machines-makers-6112/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/kettles-airpots-6113/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/juicers-blenders-grinders-6114/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/hand-stand-mixers-6115/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/ice-cream-makers-6116/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/fryers-6117/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/bbq-grills-hotpots-6118/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/breadmakers-6119/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/water-purifers-dispensers-6120/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/kitchen-appliances-30/other-kitchen-appliances-6121/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/air-conditioners-heating-6105/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/washing-machines-and-dryers-6122/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/vacuum-cleaner-housekeeping-6123/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/water-heater-instant-showers-6124/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/air-purifiers-dehumidifiers-6125/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/electrical-adaptors-sockets-6126/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/irons-steamers-6127/",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/other-home-appliances-6128/",
    "https://www.carousell.com.my/categories/babies-kids-19/",
    "https://www.carousell.com.my/categories/babies-kids-19/babies-kids-fashion-623/",
    "https://www.carousell.com.my/categories/babies-kids-19/baby-nursery-kids-furniture-6219/",
    "https://www.carousell.com.my/categories/babies-kids-19/baby-nursery-kids-furniture-6219/cots-cribs-1705/",
    "https://www.carousell.com.my/categories/babies-kids-19/baby-nursery-kids-furniture-6219/childrens-beds-6220/",
    "https://www.carousell.com.my/categories/babies-kids-19/baby-nursery-kids-furniture-6219/bed-guards-6221/",
    "https://www.carousell.com.my/categories/babies-kids-19/baby-nursery-kids-furniture-6219/kids-tables-chairs-6222/",
    "https://www.carousell.com.my/categories/babies-kids-19/baby-nursery-kids-furniture-6219/changing-tables-stations-6223/",
    "https://www.carousell.com.my/categories/babies-kids-19/baby-nursery-kids-furniture-6219/kids-wardrobes-storage-6224/",
    "https://www.carousell.com.my/categories/babies-kids-19/baby-nursery-kids-furniture-6219/nursery-lighting-decor-6225/",
    "https://www.carousell.com.my/categories/babies-kids-19/baby-nursery-kids-furniture-6219/safety-gates-locks-protectors-6226/",
    "https://www.carousell.com.my/categories/babies-kids-19/baby-nursery-kids-furniture-6219/other-kids-furniture-6227/",
    "https://www.carousell.com.my/categories/babies-kids-19/going-out-6228/",
    "https://www.carousell.com.my/categories/babies-kids-19/going-out-6228/strollers-629/",
    "https://www.carousell.com.my/categories/babies-kids-19/going-out-6228/car-seats-6229/",
    "https://www.carousell.com.my/categories/babies-kids-19/going-out-6228/carriers-slings-6230/",
    "https://www.carousell.com.my/categories/babies-kids-19/going-out-6228/diaper-bags-wetbags-6231/",
    "https://www.carousell.com.my/categories/babies-kids-19/going-out-6228/other-babies-going-out-needs-6232/",
    "https://www.carousell.com.my/categories/babies-kids-19/bathing-changing-6233/",
    "https://www.carousell.com.my/categories/babies-kids-19/bathing-changing-6233/bathtub-bath-accessories-6234/",
    "https://www.carousell.com.my/categories/babies-kids-19/bathing-changing-6233/baby-toiletries-grooming-6235/",
    "https://www.carousell.com.my/categories/babies-kids-19/bathing-changing-6233/toilet-training-6236/",
    "https://www.carousell.com.my/categories/babies-kids-19/bathing-changing-6233/diapers-baby-wipes-6237/",
    "https://www.carousell.com.my/categories/babies-kids-19/bathing-changing-6233/changing-mats-accessories-6238/",
    "https://www.carousell.com.my/categories/babies-kids-19/bathing-changing-6233/other-baby-bathing-changing-needs-6239/",
    "https://www.carousell.com.my/categories/babies-kids-19/nursing-feeding-628/",
    "https://www.carousell.com.my/categories/babies-kids-19/nursing-feeding-628/breastfeeding-bottle-feeding-6240/",
    "https://www.carousell.com.my/categories/babies-kids-19/nursing-feeding-628/weaning-toddler-feeding-6241/",
    "https://www.carousell.com.my/categories/babies-kids-19/nursing-feeding-628/baby-high-chairs-6242/",
    "https://www.carousell.com.my/categories/babies-kids-19/nursing-feeding-628/soothers-pacifiers-6243/",
    "https://www.carousell.com.my/categories/babies-kids-19/baby-monitors-6244/",
    "https://www.carousell.com.my/categories/babies-kids-19/maternity-care-627/",
    "https://www.carousell.com.my/categories/babies-kids-19/infant-playtime-626/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/toys-games-12/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/music-media-14/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/music-media-14/musical-instruments-642/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/music-media-14/music-accessories-644/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/music-media-14/cds-dvds-645/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/music-media-14/vinyls-6246/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/music-media-14/music-scores-6247/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/books-magazines-5/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/books-magazines-5/storybooks-588/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/books-magazines-5/textbooks-15/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/books-magazines-5/assessment-books-6248/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/books-magazines-5/childrens-books-590/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/books-magazines-5/comics-manga-591/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/books-magazines-5/magazines-594/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/books-magazines-5/travel-holiday-guides-6249/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/books-magazines-5/religion-books-6250/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/stationery-craft-8/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/stationery-craft-8/art-prints-676/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/stationery-craft-8/handmade-craft-674/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/stationery-craft-8/flowers-bouquets-6251/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/stationery-craft-8/craft-supplies-and-tools-673/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/stationery-craft-8/stationery-school-supplies-592/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/stationery-craft-8/occasions-party-supplies-6252/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/stationery-craft-8/other-stationery-craft-696/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/collectibles-memorabilia-9/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/collectibles-memorabilia-9/stamps-prints-648/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/collectibles-memorabilia-9/religious-items-750/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/collectibles-memorabilia-9/currency-647/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/collectibles-memorabilia-9/vintage-collectibles-650/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/collectibles-memorabilia-9/fan-merchandise-6253/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/collectibles-memorabilia-9/j-pop-31/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/collectibles-memorabilia-9/k-wave-25/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/travel-1951/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/travel-1951/luggages-1952/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/travel-1951/travel-essentials-accessories-1954/",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/travel-1951/umbrellas-6254/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/insect-repellent-6268/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/massage-devices-6269/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/assistive-rehabilatory-aids-6270/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/assistive-rehabilatory-aids-6270/visual-hearing-aids-6271/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/assistive-rehabilatory-aids-6270/wheelchairs-6272/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/assistive-rehabilatory-aids-6270/rehabilitative-devices-6273/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/assistive-rehabilatory-aids-6270/other-assistive-aids-6274/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/assistive-rehabilatory-aids-6270/adult-incontinence-6275/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/face-masks-face-shields-6276/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/thermometers-6277/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/medical-supplies-tools-6278/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/health-supplements-6279/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/health-supplements-6279/health-food-drinks-tonics-6280/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/health-supplements-6279/vitamins-supplements-6281/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/health-supplements-6279/sports-fitness-nutrition-6282/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/braces-support-protection-6283/",
    "https://www.carousell.com.my/categories/health-nutrition-6267/health-monitors-weighing-scales-6284/",
    "https://www.carousell.com.my/categories/sports-equipment-10/",
    "https://www.carousell.com.my/categories/sports-equipment-10/sports-games-6285/",
    "https://www.carousell.com.my/categories/sports-equipment-10/sports-games-6285/racket-ball-sports-6286/",
    "https://www.carousell.com.my/categories/sports-equipment-10/sports-games-6285/skates-rollerblades-scooters-716/",
    "https://www.carousell.com.my/categories/sports-equipment-10/sports-games-6285/water-sports-6287/",
    "https://www.carousell.com.my/categories/sports-equipment-10/sports-games-6285/golf-6288/",
    "https://www.carousell.com.my/categories/sports-equipment-10/sports-games-6285/billiards-bowling-6289/",
    "https://www.carousell.com.my/categories/sports-equipment-10/sports-games-6285/kites-6290/",
    "https://www.carousell.com.my/categories/sports-equipment-10/exercise-fitness-6291/",
    "https://www.carousell.com.my/categories/sports-equipment-10/exercise-fitness-6291/weights-dumbells-6292/",
    "https://www.carousell.com.my/categories/sports-equipment-10/exercise-fitness-6291/cardio-fitness-machines-6293/",
    "https://www.carousell.com.my/categories/sports-equipment-10/exercise-fitness-6291/toning-stretching-accessories-6294/",
    "https://www.carousell.com.my/categories/sports-equipment-10/exercise-fitness-6291/exercise-mats-6295/",
    "https://www.carousell.com.my/categories/sports-equipment-10/bicycles-parts-711/",
    "https://www.carousell.com.my/categories/sports-equipment-10/bicycles-parts-711/bicycles-6296/",
    "https://www.carousell.com.my/categories/sports-equipment-10/fishing-6297/",
    "https://www.carousell.com.my/categories/sports-equipment-10/hiking-camping-1953/",
    "https://www.carousell.com.my/categories/sports-equipment-10/other-sports-equipment-and-supplies-736/",
    "https://www.carousell.com.my/categories/snacks-1210/",
    "https://www.carousell.com.my/categories/snacks-1210/local-eats-2160/",
    "https://www.carousell.com.my/categories/snacks-1210/rice-noodles-6298/",
    "https://www.carousell.com.my/categories/snacks-1210/homemade-bakes-1253/",
    "https://www.carousell.com.my/categories/snacks-1210/beverages-1221/",
    "https://www.carousell.com.my/categories/snacks-1210/packaged-instant-food-1219/",
    "https://www.carousell.com.my/categories/snacks-1210/fresh-produce-6299/",
    "https://www.carousell.com.my/categories/snacks-1210/spice-seasoning-6300/",
    "https://www.carousell.com.my/categories/snacks-1210/chilled-frozen-food-6301/",
    "https://www.carousell.com.my/categories/snacks-1210/gift-baskets-hampers-6302/",
    "https://www.carousell.com.my/categories/snacks-1210/other-food-drinks-2217/",
    "https://www.carousell.com.my/categories/auto-accessories-109/",
    "https://www.carousell.com.my/categories/motorbikes-108/",
    "https://www.carousell.com.my/categories/pet-supplies-29/",
    "https://www.carousell.com.my/categories/pet-supplies-29/homes-other-pet-accessories-738/",
    "https://www.carousell.com.my/categories/pet-supplies-29/pet-food-740/",
    "https://www.carousell.com.my/categories/pet-supplies-29/health-grooming-6303/",
    "https://www.carousell.com.my/categories/cars-32/",
    "https://www.carousell.com.my/categories/cars-32/cars-for-sale-632/",
    "https://www.carousell.com.my/categories/cars-32/vehicle-rentals-634/",
    "https://www.carousell.com.my/categories/tickets-vouchers-18/",
    "https://www.carousell.com.my/categories/tickets-vouchers-18/local-attractions-and-transport-747/",
    "https://www.carousell.com.my/categories/tickets-vouchers-18/vouchers-748/",
    "https://www.carousell.com.my/categories/tickets-vouchers-18/events-tickets-749/",
    "https://www.carousell.com.my/categories/tickets-vouchers-18/store-credits-6304/",
    "https://www.carousell.com.my/categories/tickets-vouchers-18/flights-overseas-attractions-1950/",
    "https://www.carousell.com.my/categories/property-102/",
    "https://www.carousell.com.my/categories/property-102/housing-for-sale-636/",
    "https://www.carousell.com.my/categories/property-102/property-rentals-635/",
    "https://www.carousell.com.my/categories/property-102/others-property-637/",
    "https://www.carousell.com.my/categories/services-1426/",
    "https://www.carousell.com.my/categories/services-1426/home-services-1427/",
    "https://www.carousell.com.my/categories/services-1426/home-services-1427/cleaning-1428/",
    "https://www.carousell.com.my/categories/services-1426/home-services-1427/aircon-services-1429/",
    "https://www.carousell.com.my/categories/services-1426/home-services-1427/electrical-lighting-wiring-1430/",
    "https://www.carousell.com.my/categories/services-1426/home-services-1427/home-repairs-1431/",
    "https://www.carousell.com.my/categories/services-1426/home-services-1427/movers-delivery-1432/",
    "https://www.carousell.com.my/categories/services-1426/home-services-1427/renovations-1433/",
    "https://www.carousell.com.my/categories/services-1426/home-services-1427/others-1434/",
    "https://www.carousell.com.my/categories/services-1426/electronics-gadget-repairs-1435/",
    "https://www.carousell.com.my/categories/services-1426/tuition-1436/",
    "https://www.carousell.com.my/categories/services-1426/beauty-services-1437/",
    "https://www.carousell.com.my/categories/services-1426/others-1438/",
    "https://www.carousell.com.my/categories/jobs-1439/",
    "https://www.carousell.com.my/categories/jobs-1439/part-time-1440/",
    "https://www.carousell.com.my/categories/jobs-1439/part-time-1440/hospitality-f-b-1441/",
    "https://www.carousell.com.my/categories/jobs-1439/part-time-1440/sales-retail-marketing-1442/",
    "https://www.carousell.com.my/categories/jobs-1439/part-time-1440/events-1443/",
    "https://www.carousell.com.my/categories/jobs-1439/part-time-1440/customer-service-1444/",
    "https://www.carousell.com.my/categories/jobs-1439/part-time-1440/security-1445/",
    "https://www.carousell.com.my/categories/jobs-1439/part-time-1440/warehouse-logistics-1446/",
    "https://www.carousell.com.my/categories/jobs-1439/part-time-1440/admin-office-finance-1447/",
    "https://www.carousell.com.my/categories/jobs-1439/part-time-1440/cleaning-housekeeping-1448/",
    "https://www.carousell.com.my/categories/jobs-1439/part-time-1440/drivers-delivery-1449/",
    "https://www.carousell.com.my/categories/jobs-1439/part-time-1440/computer-it-1450/",
    "https://www.carousell.com.my/categories/jobs-1439/part-time-1440/others-1451/",
    "https://www.carousell.com.my/categories/jobs-1439/full-time-1452/",
    "https://www.carousell.com.my/categories/jobs-1439/full-time-1452/hospitality-f-b-1453/",
    "https://www.carousell.com.my/categories/jobs-1439/full-time-1452/sales-retail-marketing-1454/",
    "https://www.carousell.com.my/categories/jobs-1439/full-time-1452/events-1455/",
    "https://www.carousell.com.my/categories/jobs-1439/full-time-1452/customer-service-1456/",
    "https://www.carousell.com.my/categories/jobs-1439/full-time-1452/security-1457/",
    "https://www.carousell.com.my/categories/jobs-1439/full-time-1452/warehouse-logistics-1458/",
    "https://www.carousell.com.my/categories/jobs-1439/full-time-1452/admin-office-finance-1459/",
    "https://www.carousell.com.my/categories/jobs-1439/full-time-1452/cleaning-housekeeping-1460/",
    "https://www.carousell.com.my/categories/jobs-1439/full-time-1452/drivers-delivery-1461/",
    "https://www.carousell.com.my/categories/jobs-1439/full-time-1452/computer-it-1462/",
    "https://www.carousell.com.my/categories/jobs-1439/full-time-1452/others-1463/",
    "https://www.carousell.com.my/categories/jobs-1439/internships-others-1478/",
    "https://www.carousell.com.my/categories/community-26/",
    "https://www.carousell.com.my/categories/looking-for-21/",
    "https://www.carousell.com.my/categories/everything-else-16/",
    "https://www.carousell.com.my/categories/everything-else-16/others-everything-else-1185/",
    "https://www.carousell.com.my/categories/announcements-6306/",
]

print(f"Total links: {len(URLS)}")

# ── Config ─────────────────────────────────────────────────────────────────────
CSV_FILE        = "carousell_data_adam.csv"
PROGRESS_FILE   = "carousell_progress_adam.txt"
MAX_RECORDS     = 200_000
SAVE_EVERY      = 500
SCROLL_PAUSE    = 1.5        # reduced from 2.0
MAX_STALE       = 2
NUM_WORKERS     = 3          # ← tune this (2–4 is safe; more = higher ban risk)

# Pre-compiled regex (module-level, not inside hot loops)
CATEGORY_ID_RE = re.compile(r'-\d+')

# ── Shared state (thread-safe) ─────────────────────────────────────────────────
results_lock    = threading.Lock()
seen_ids_lock   = threading.Lock()
progress_lock   = threading.Lock()
stop_event      = threading.Event()   # set when MAX_RECORDS is reached

results         = []
seen_ids        = set()
completed_urls  = set()

# ── Resume: load existing CSV & completed-URL log ──────────────────────────────
if os.path.exists(CSV_FILE):
    print(f"📂 Found existing file – resuming from '{CSV_FILE}'…")
    df_old  = pd.read_csv(CSV_FILE)
    results = df_old.to_dict(orient="records")
    if "listing_id" in df_old.columns:
        seen_ids = set(df_old["listing_id"].dropna().astype(str))
        print(f"   ↳ Resumed with {len(seen_ids):,} previously scraped cards.")
    else:
        print("   ⚠️  Old CSV missing 'listing_id' – duplicates may occur.")
else:
    print("🚀 Starting fresh scrape.")

if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, "r") as f:
        completed_urls = set(line.strip() for line in f if line.strip())
    print(f"   ↳ {len(completed_urls)} URLs already completed – will skip them.\n")

# Track last-save threshold (protected by results_lock)
_last_save_count = len(results)


# ── Persistence helpers ────────────────────────────────────────────────────────
def save():
    """Thread-safe CSV save."""
    global _last_save_count
    with results_lock:
        pd.DataFrame(results).to_csv(CSV_FILE, index=False)
        _last_save_count = len(results)
        print(f"   💾 [{threading.current_thread().name}] "
              f"Saved {len(results):,} records → {CSV_FILE}")


def mark_url_done(url: str):
    """Thread-safe progress log."""
    with progress_lock:
        with open(PROGRESS_FILE, "a") as f:
            f.write(url + "\n")
        completed_urls.add(url)


# ── Browser factory ────────────────────────────────────────────────────────────

_driver_init_lock = threading.Lock()

def make_driver() -> uc.Chrome:
    """Create a fresh headless Chrome instance for one worker thread."""
    with _driver_init_lock:
        opts = uc.ChromeOptions()
        opts.add_argument("--headless=new")
        opts.add_argument("--blink-settings=imagesEnabled=false")
        opts.add_argument("--no-sandbox")
        opts.add_argument("--disable-dev-shm-usage")
        opts.add_argument("--disable-extensions")
        opts.add_argument("--disable-gpu")
        opts.add_argument(
            "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
        )
        # Stagger port so multiple instances don't clash
        return uc.Chrome(options=opts, version_main=147)


# ── Parsing helpers (stateless – safe to call from any thread) ─────────────────
def get_category_from_url(url: str) -> str:
    """Derive category from URL (no driver needed – fast fallback)."""
    try:
        path = url.split("/categories/")[1].split("?")[0]
        parts = CATEGORY_ID_RE.sub("", path).strip("-").split("/")
        category = " > ".join(p.strip().replace("-", " ") for p in parts if p.strip())
        return category if category else "Unknown"
    except Exception:
        return "Unknown"


def get_category_from_page(driver) -> str:
    """Try breadcrumb first; fall back to URL-derived name."""
    try:
        breadcrumb_ul = driver.find_element(By.CSS_SELECTOR, "ul.D_aI_")
        lis = breadcrumb_ul.find_elements(By.TAG_NAME, "li")
        texts = [li.text.strip() for li in lis[1:] if li.text.strip()]
        if texts:
            return " > ".join(texts)
    except Exception:
        pass
    return get_category_from_url(driver.current_url)


def parse_cards(page_source: str, current_url: str, category_text: str) -> list[dict]:
    """
    Parse listing cards from raw HTML.
    Returns only records whose listing_id is NOT already in seen_ids.
    Does NOT mutate seen_ids – caller does that after acquiring the lock.
    """
    soup = BeautifulSoup(page_source, "lxml")   # lxml is 2-5x faster than html.parser

    cards = soup.select(
        '[data-testid^="listing-card-"]'
        ':not([data-testid="listing-card-text-seller-name"])'
        ':not([data-testid="listing-card-btn-like"])'
    )

    # Snapshot seen_ids once (avoids repeated lock acquisition inside the loop)
    with seen_ids_lock:
        local_seen = frozenset(seen_ids)

    new_records = []
    new_ids     = []

    for card in cards:
        raw_testid = card.get("data-testid", "")
        listing_id = raw_testid.replace("listing-card-", "")

        if not listing_id or listing_id in local_seen or listing_id in new_ids:
            continue

        img_el  = card.select_one('a[href^="/p/"] img')
        product = img_el["alt"].strip() if img_el and img_el.get("alt") else None
        if not product:
            continue

        link_el = card.select_one('a[href^="/p/"]')
        listing_url = (
            "https://www.carousell.com.my" + link_el["href"].split("?")[0]
            if link_el else None
        )

        price_el       = card.select_one('p[title^="RM"]')
        price          = price_el["title"].strip() if price_el else None

        orig_el        = card.select_one('span[aria-label^="Stricken Price"] s')
        original_price = orig_el.get_text(strip=True) if orig_el else None

        condition = None
        if price_el:
            condition_el = price_el.parent.find_next_sibling("p")
            if condition_el:
                condition = condition_el.text.strip()

        seller_el      = card.select_one('[data-testid="listing-card-text-seller-name"]')
        seller         = seller_el.text.strip() if seller_el else None

        time_el = next(
            (p for p in card.select("p")
            if p.text and re.search(
                r'\d+\s+(second|minute|hour|day|week|month|year)s?\s+ago',
                p.text, re.I
            )),
            None
        )
        time_posted = time_el.text.strip() if time_el else None

        like_el        = card.select_one('[data-testid="listing-card-btn-like"] span')
        likes          = like_el.text.strip() if like_el else "0"

        bp_badge       = card.select_one('div.D_uT p')
        buyer_protection = bool(bp_badge and 'Buyer Protection' in bp_badge.get_text())

        # Find a <p> mentioning delivery/shipping keywords
        delivery_el = next(
            (p for p in card.select("p")
            if p.text and re.search(r'deliver|shipping|courier|free ship', p.text, re.I)),
            None
        )
        delivery_info = delivery_el.get_text(strip=True) if delivery_el else None
        
        new_ids.append(listing_id)
        new_records.append({
            "listing_id":       listing_id,
            "product":          product,
            "price":            price,
            "original_price":   original_price,
            "condition":        condition,
            "seller":           seller,
            "time_posted":      time_posted,
            "likes":            likes,
            "listing_url":      listing_url,
            "source_url":       current_url,
            "category":         category_text,
            "buyer_protection": buyer_protection,
            "delivery_info":    delivery_info,
        })

    return new_records, new_ids


# ── Per-URL scraper (runs inside a worker thread) ──────────────────────────────
def scrape_url(driver, url: str, worker_name: str) -> int:
    """
    Scrape one category URL with the given driver.
    Returns the number of new records added.
    """
    global _last_save_count

    print(f"\n[{worker_name}] 🌐 {url}")

    driver.get(url)
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, '[data-testid^="listing-card-"]')
            )
        )
    except TimeoutException:
        print(f"[{worker_name}] ❌ Timed out – skipping.")
        return 0

    time.sleep(1.5)   # reduced settle time

    category_text = get_category_from_page(driver)
    print(f"[{worker_name}] 📁 {category_text}")

    stale_count    = 0
    total_added    = 0

    while not stop_event.is_set():

        new_records, new_ids = parse_cards(
            driver.page_source, driver.current_url, category_text
        )

        if new_records:
            with seen_ids_lock:
                # Re-check under lock to avoid races between workers
                truly_new = [
                    (rec, lid)
                    for rec, lid in zip(new_records, new_ids)
                    if lid not in seen_ids
                ]
                for _, lid in truly_new:
                    seen_ids.add(lid)

            if truly_new:
                records_only = [r for r, _ in truly_new]
                with results_lock:
                    results.extend(records_only)
                    current_total = len(results)
                    should_save   = (current_total - _last_save_count) >= SAVE_EVERY

                total_added += len(records_only)
                stale_count  = 0
                print(f"[{worker_name}] 📊 +{len(records_only)} "
                      f"(total: {current_total:,})")

                if should_save:
                    save()

                if current_total >= MAX_RECORDS:
                    stop_event.set()
                    print(f"[{worker_name}] 🏁 MAX_RECORDS reached – signalling stop.")
                    break
            else:
                stale_count += 1
        else:
            stale_count += 1

        if stale_count >= MAX_STALE:
            print(f"[{worker_name}] ✅ Stale limit – done with this URL.")
            break

        if stale_count:
            print(f"[{worker_name}] ⏳ No new cards ({stale_count}/{MAX_STALE})")

        # Click "Show more results" if available
        try:
            show_more = driver.find_element(
                By.XPATH, "//button[contains(., 'Show more results')]"
            )
            if show_more.is_displayed():
                driver.execute_script("arguments[0].scrollIntoView(true);", show_more)
                time.sleep(0.4)
                show_more.click()
                print(f"[{worker_name}] 👉 Clicked 'Show more results'")
                time.sleep(SCROLL_PAUSE)
                continue   # re-parse immediately after click
        except Exception:
            pass

        # Scroll to bottom
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(SCROLL_PAUSE)
        driver.execute_script("window.scrollBy(0, -300);")
        time.sleep(0.2)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.4)

    return total_added


# ── Worker: owns one browser, drains its URL queue ────────────────────────────
def worker(url_queue: list[str], worker_id: int):
    """
    Each worker thread gets a dedicated Chrome instance and a slice of URLs.
    """
    name = f"W{worker_id}"
    print(f"[{name}] 🚀 Starting browser…")
    driver = make_driver()
    print(f"[{name}] ✅ Browser ready. {len(url_queue)} URLs assigned.")

    try:
        for url in url_queue:
            if stop_event.is_set():
                break

            # Skip if another worker already completed this URL between start and now
            with progress_lock:
                if url in completed_urls:
                    print(f"[{name}] ⏭  Already done – {url}")
                    continue

            added = scrape_url(driver, url, name)
            mark_url_done(url)
            print(f"[{name}] ✔ Finished URL (+{added} records): {url}")
            time.sleep(2)   # polite delay between URLs within a worker

    finally:
        driver.quit()
        print(f"[{name}] 🛑 Browser closed.")


# ── URL distribution ──────────────────────────────────────────────────────────
def distribute(urls: list[str], n: int) -> list[list[str]]:
    """
    Round-robin distribution so each worker gets a balanced mix of
    light (subcategory) and heavy (top-level) URLs.
    """
    buckets = [[] for _ in range(n)]
    for i, url in enumerate(urls):
        buckets[i % n].append(url)
    return buckets


# ── Main ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    urls_to_scrape = [u for u in URLS if u not in completed_urls]
    print(f"\n📋 {len(urls_to_scrape)} URLs to scrape across {NUM_WORKERS} workers "
          f"({len(completed_urls)} already done).\n")

    if not urls_to_scrape:
        print("Nothing to do – all URLs already completed.")
        sys.exit(0)

        print("🔧 Preloading undetected_chromedriver...")

    try:
        temp_driver = make_driver()
        temp_driver.quit()
        print("✅ Driver preloaded successfully.")
    except Exception as e:
        print(f"❌ Failed to preload driver: {e}")
        sys.exit(1)

    # Clamp workers to number of URLs (no point spinning up idle browsers)
    n_workers = min(NUM_WORKERS, len(urls_to_scrape))
    url_buckets = distribute(urls_to_scrape, n_workers)

    with ThreadPoolExecutor(max_workers=n_workers,
                            thread_name_prefix="Scraper") as pool:
        futures = {
            pool.submit(worker, bucket, wid): wid
            for wid, bucket in enumerate(url_buckets, 1)
        }
        for future in as_completed(futures):
            wid = futures[future]
            try:
                future.result()
            except Exception as exc:
                print(f"[W{wid}] ⚠️  Worker crashed: {exc}")

    # Final save
    save()
    print(f"\n🏁 All done. {len(results):,} records saved to '{CSV_FILE}'.")

Total links: 545
📂 Found existing file – resuming from 'carousell_data_adam.csv'…


C:\Users\user\AppData\Local\Temp\ipykernel_19564\1237856514.py:594: DtypeWarning: Columns (0: likes) have mixed types. Specify dtype option on import or set low_memory=False.
  df_old  = pd.read_csv(CSV_FILE)


   ↳ Resumed with 182,607 previously scraped cards.
   ↳ 251 URLs already completed – will skip them.


📋 394 URLs to scrape across 3 workers (251 already done).

✅ Driver preloaded successfully.
[W1] 🚀 Starting browser…
[W2] 🚀 Starting browser…
[W3] 🚀 Starting browser…
[W1] ✅ Browser ready. 132 URLs assigned.

[W1] 🌐 https://www.carousell.com.my/categories/women-s-fashion-4/dresses-sets-1259/jumpsuits-6141/
[W2] ✅ Browser ready. 131 URLs assigned.

[W2] 🌐 https://www.carousell.com.my/categories/women-s-fashion-4/dresses-sets-1259/traditional-ethnic-wear-6143/
[W3] ✅ Browser ready. 131 URLs assigned.

[W3] 🌐 https://www.carousell.com.my/categories/women-s-fashion-4/footwear-612/boots-6144/
[W1] 📁 women s fashion > dresses sets > jumpsuits
[W1] 📊 +5 (total: 182,612)
[W3] 📁 women s fashion > footwear > boots
[W2] 📁 women s fashion > dresses sets > traditional ethnic wear
[W3] 📊 +1 (total: 182,613)
[W2] 📊 +31 (total: 182,644)
[W1] 👉 Clicked 'Show more results'
[W3] 👉 Clicked 'Show more re